In [ ]:
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.svm import LinearSVC


# ============================================================================
# 04. SVM TRAINING AND FULL EVALUATION
# ============================================================================

BASE_DIR = Path("..").resolve()
MATRICES_PATH = BASE_DIR / "data/processed/nli_feature_matrices.pkl"
MODELS_DIR = BASE_DIR / "models"
RESULTS_DIR = BASE_DIR / "data/processed/results"

MODEL_PATH = MODELS_DIR / "linear_svc_nli_model.pkl"
PREDICTIONS_PATH = RESULTS_DIR / "svm_predictions_full.csv"
SUMMARY_PATH = RESULTS_DIR / "svm_metrics_full.csv"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

with MATRICES_PATH.open("rb") as f:
    data = pickle.load(f)

X = data["X"]
y = np.asarray(data["y"])
doc_id = np.asarray(data["doc_id"])
dataset_tags = np.asarray(data["dataset_tags"])
split_tags = np.asarray(data["split_tags"])

train_mask = (dataset_tags == "TOEFL11") & (split_tags == "train")
toefl_test_mask = (dataset_tags == "TOEFL11") & (split_tags == "test")
icle_test_mask = dataset_tags == "ICLEV3"

X_train, y_train = X[train_mask], y[train_mask]

print(f"Training rows: {X_train.shape[0]}")
print(f"TOEFL11 test rows: {toefl_test_mask.sum()}")
print(f"ICLEv3 external test rows: {icle_test_mask.sum()}")

model = LinearSVC(C=1.0, random_state=42, max_iter=10000)

with warnings.catch_warnings(record=True) as caught_warnings:
    warnings.simplefilter("always", ConvergenceWarning)
    model.fit(X_train, y_train)

convergence_warnings = [w for w in caught_warnings if issubclass(w.category, ConvergenceWarning)]
if convergence_warnings:
    print("Warning: LinearSVC emitted a convergence warning. Consider increasing max_iter or tuning C.")


def evaluate_partition(mask: np.ndarray, partition_name: str) -> tuple[pd.DataFrame, dict]:
    X_eval = X[mask]
    y_true = y[mask]
    y_pred = model.predict(X_eval)

    pred_df = pd.DataFrame(
        {
            "doc_id": doc_id[mask],
            "dataset": dataset_tags[mask],
            "split": split_tags[mask],
            "true_l1": y_true,
            "svm_predicted_l1": y_pred,
            "partition": partition_name,
        }
    )

    metrics = {
        "partition": partition_name,
        "n": len(y_true),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
    }

    print(f"\n=== {partition_name} ===")
    print(pd.Series(metrics))
    print(classification_report(y_true, y_pred, digits=4))

    labels = sorted(np.unique(y))
    cm = pd.DataFrame(
        confusion_matrix(y_true, y_pred, labels=labels),
        index=[f"true_{x}" for x in labels],
        columns=[f"pred_{x}" for x in labels],
    )
    cm.to_csv(RESULTS_DIR / f"svm_confusion_{partition_name}.csv")

    return pred_df, metrics


toefl_predictions, toefl_metrics = evaluate_partition(toefl_test_mask, "toefl11_test")
icle_predictions, icle_metrics = evaluate_partition(icle_test_mask, "iclev3_external_test")

all_predictions = pd.concat([toefl_predictions, icle_predictions], ignore_index=True)
all_predictions.to_csv(PREDICTIONS_PATH, index=False)
pd.DataFrame([toefl_metrics, icle_metrics]).to_csv(SUMMARY_PATH, index=False)

with MODEL_PATH.open("wb") as f:
    pickle.dump(model, f)

print(f"Saved model: {MODEL_PATH}")
print(f"Saved predictions: {PREDICTIONS_PATH}")
print(f"Saved metrics: {SUMMARY_PATH}")
